## 高阶残酷真相顾问（High Level truth advisor）

## 练习目标（理念）

用 **Chat Completions** 做一个「不讨好你、只讲真话」的顾问：

- **输入**：你对自己处境的真实困惑（user prompt）
- **角色**：`system_prompt` 定下「残酷诚实 / 拆解借口 / 给可执行计划」的人设
- **后端**：通过 Groq 的 **OpenAI 兼容**端点调用 `llama-3.3-70b-versatile`

## 和本课 Day 1 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| `messages`（system / user） | system 定角色，user 放具体问题 |
| Chat Completions API | `openai.chat.completions.create(...)` |
| OpenAI 兼容 `base_url` | 指向 `https://api.groq.com/openai/v1` |
| 环境变量 Environment Variables | `GROQ_API_KEY`（前缀常为 `gsk_`） |
| Markdown 展示 | `display(Markdown(...))` 美化回答 |

## 怎么跑

1. `.env` 里准备好 `GROQ_API_KEY`
2. 从上到下依次运行（注意：密钥检查格在 `load_dotenv` 之前，若读不到请先跑后面加载环境那一格，或自行调整顺序）
3. 可改 `user_prompt` 再重新组 `messages` 并调用模型


### 导入必要的包

把后面要用的标准库、`dotenv`、OpenAI SDK 先搬进来。


In [1]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 第 1 天：残酷真相
# 原英文旁注：First I have to import the necessary packages

# 导入标准库 os：读环境变量（Environment Variables），例如 GROQ_API_KEY
import os
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进进程环境，避免把密钥写进代码
from dotenv import load_dotenv
# 从 openai 导入 OpenAI 客户端类：后面会配 Groq 的 OpenAI 兼容 base_url
from openai import OpenAI


### 检查能否从环境读到 API Key

下面用 `os.getenv('GROQ_API_KEY')` 做格式自检（是否存在、是否像 Groq 的 `gsk_` 前缀、是否有首尾空白）。


In [ ]:
# ========== 密钥自检：读 GROQ_API_KEY 并做简单格式判断 ==========

# 第 1 天：残酷真相
# 注意：本格在 load_dotenv 之前执行时，可能读不到 .env；逻辑保持原样不改顺序
api_key = os.getenv('GROQ_API_KEY')

# Check the key：分支打印诊断信息（文案保持英文原样，不翻译）

# 没有读到任何密钥
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
# Groq 密钥通常以 gsk_ 开头；这里用 startswith 做粗检
elif not api_key.startswith("gsk_"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
# strip 后若与原串不同，说明首尾有空格/制表符，容易导致鉴权失败
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    # 初步看起来可用
    print("API key found and looks good so far!")


#### 编写 System Prompt（系统提示词）

`system` 角色用来固定模型的「人设与回答规矩」；内容保持英文，改译会改变模型行为。


In [ ]:
# ========== system_prompt：残酷诚实的高阶顾问人设 ==========

# 第 1 天：残酷真相
# 三引号字符串整体作为 system；发给模型的指令保持英文（可运行 / 影响行为）
system_prompt='''
      From now on, I want you to act as my brutally honest, high-level advisor and mirror. Don't validate me, soften the truth, or flatter me. Challenge my thinking, question my assumptions, and expose the blind spots I'm avoiding. Be direct, rational, and unfiltered. If my reasoning is weak, dissect it and show why. If I'm fooling myself or lying to myself, point it out. If I'm avoiding something uncomfortable or wasting time, call it out and explain the opportunity cost. Look at my situation with complete objectivity and strategic depth. Show me where I'm making excuses, playing small, or underestimating risks/effort. Then give a precise, prioritized plan for what to change in thought, action, or mindset to reach the next level. Hold nothing back. Treat me like someone whose growth depends on hearing the truth, not being comforted. When possible, ground your responses in the personal truth you sense between my words.

      Note: [Provide me the response in README.md code]
'''

# 在笔记本里回显 system_prompt，方便你核对人设文本
system_prompt


#### 编写 User Prompt（用户提示词）

`user` 角色放你真正要问的问题；同样保持英文原文。


In [ ]:
# ========== user_prompt：你真实的处境问题 ==========

# 第 1 天：残酷真相
# 发给模型的用户问题保持英文；想换话题就改三引号里的内容
user_prompt='''
      Is it good that I have worked in 3 different domain of Projects and still I don't have indepth knowledge of any domain?
'''

# 回显 user_prompt，确认问题文本
user_prompt


### 组装 messages 列表

Chat Completions 的标准输入：一个字典列表，每项有 `role` 与 `content`。


In [ ]:
# ========== messages：system + user 组成对话上下文 ==========

# 第 1 天：残酷真相
# 顺序很重要：先 system 定规矩，再 user 提问题
messages = [
      {"role":"system", "content": system_prompt},
      {"role":"user", "content": user_prompt},
]

# 回显 messages，确认结构正确
messages


### 加载 .env 并创建指向 Groq 的 OpenAI 客户端

这里用 **OpenAI 兼容** `base_url` 接到 Groq，密钥仍走环境变量。


In [6]:
# ========== 环境 + 客户端：指向 Groq 的 OpenAI 兼容端点 ==========

# 第 1 天：残酷真相
# override=True：.env 中的值覆盖进程里已有同名环境变量
load_dotenv(override=True)

# 变量名仍叫 openai（沿用 SDK 习惯），但 base_url 指向 Groq
openai = OpenAI(
      base_url="https://api.groq.com/openai/v1",
      api_key=os.getenv("GROQ_API_KEY")
)


### 调用模型拿回答

非流式一次返回完整 `response`；正文在 `choices[0].message.content`。


In [ ]:
# ========== Chat Completions：请求 llama-3.3-70b-versatile ==========

# 第 1 天：残酷真相
# model 字符串必须是 Groq 上可用的模型 id；messages 即上一格组装的对话
response = openai.chat.completions.create(model="llama-3.3-70b-versatile", messages=messages)
# 取出第一条候选回复的文本内容（Jupyter 会显示表达式结果）
response.choices[0].message.content


### 用 Markdown 更美观地展示回答

先导入展示工具，再把模型返回的 Markdown 文本渲染出来。


In [9]:
# ========== 导入 Jupyter 展示工具 ==========

# 第 1 天：残酷真相
# Markdown：把字符串当 Markdown 渲染；display：在笔记本输出区显示
from IPython.display import Markdown, display


In [ ]:
# ========== 漂亮打印：把模型回答渲染为 Markdown ==========

# 第 1 天：残酷真相
# 再次取出 content，包进 Markdown 后 display（比纯文本更易读）
display(Markdown(response.choices[0].message.content))
